In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Let's explore the repository structure
import os

repo_root = '/net/scratch2/smallyan/function_vectors_eval'

def list_files_recursive(path, prefix='', max_depth=3, current_depth=0):
    """List files recursively with indentation"""
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(os.listdir(path))
    except PermissionError:
        return
    
    for item in items:
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            print(f"{prefix}{item}/")
            list_files_recursive(item_path, prefix + '  ', max_depth, current_depth + 1)
        else:
            print(f"{prefix}{item}")

print(f"Repository root: {repo_root}")
print("\nRepository structure:")
list_files_recursive(repo_root)

Repository root: /net/scratch2/smallyan/function_vectors_eval

Repository structure:
.git/
  COMMIT_EDITMSG
  FETCH_HEAD
  HEAD
  config
  description
  hooks/
    applypatch-msg.sample
    commit-msg.sample
    fsmonitor-watchman.sample
    post-update.sample
    pre-applypatch.sample
    pre-commit.sample
    pre-merge-commit.sample
    pre-push.sample
    pre-rebase.sample
    pre-receive.sample
    prepare-commit-msg.sample
    push-to-checkout.sample
    sendemail-validate.sample
    update.sample
  index
  info/
    exclude
  logs/
    HEAD
    refs/
  objects/
    05/
    0d/
    10/
    11/
    15/
    19/
    1a/
    1b/
    1c/
    1d/
    25/
    27/
    2d/
    2e/
    2f/
    35/
    36/
    39/
    3c/
    42/
    45/
    48/
    4c/
    4e/
    52/
    53/
    58/
    5a/
    5b/
    5d/
    5f/
    64/
    6b/
    75/
    7a/
    7d/
    7e/
    81/
    82/
    87/
    8e/
    90/
    92/
    94/
    98/
    9b/
    9c/
    a0/
    a3/
    a5/
    aa/
    ae/
    af/
  

In [3]:
# Read the plan.md file
with open(os.path.join(repo_root, 'plan.md'), 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across diverse ICL tasks. 

In [4]:
# Read the CodeWalkthrough.md file
with open(os.path.join(repo_root, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# Function Vectors in Large Language Models
### [Project Website](https://functions.baulab.info) | [Arxiv Preprint](https://arxiv.org/abs/2310.15213) | [OpenReview](https://openreview.net/forum?id=AwyxtyMwaG)

This repository contains data and code for the paper: [Function Vectors in Large Language Models](https://arxiv.org/pdf/2310.15213).

<p align="left">
<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" style="width:100%;"/>
</p> 

## Setup

We recommend using conda as a package manager. 
The environment used for this project can be found in the `fv_environment.yml` file.
To install, you can run: 
```
conda env create -f fv_environment.yml
conda activate fv
```

## Demo Notebook
Checkout `notebooks/fv_demo.ipynb` for a jupyter notebook with a demo of how to create a function vector and use it in different contexts.

## Data
The datasets used in our project can be found in the `dataset_files` folder.

## Code
Our main evaluation scripts are contained in the

In [5]:
# Read the demo notebook to understand the core implementation
with open(os.path.join(repo_root, 'notebooks/fv_demo.ipynb'), 'r') as f:
    import json
    demo_nb = json.load(f)

# Print the cells
for i, cell in enumerate(demo_nb['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    print(f"=== Cell {i} ({cell_type}) ===")
    print(source)
    print()

=== Cell 0 (code) ===
%load_ext autoreload
%autoreload 2

=== Cell 1 (code) ===
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

=== Cell 2 (markdown) ===
## Load model & tokenizer

=== Cell 3 (code) ===
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9

=== Cell 4 (markdown) ===
## Load dataset and Compute task-conditioned mean activations

=== Cell 5 (code) ===
dataset = load_dataset('antonym', seed=0)
mean_activations = get_mean_head_activati

In [6]:
# Let's read the key utility files to understand the implementation
# First, read extract_utils.py
with open(os.path.join(repo_root, 'src/utils/extract_utils.py'), 'r') as f:
    extract_utils_content = f.read()
print("=== extract_utils.py ===")
print(extract_utils_content[:5000])

=== extract_utils.py ===
import os, re, json

import torch, numpy as np
import pandas as pd
from baukit import TraceDict

# Include prompt creation helper functions
from .prompt_utils import *
from .intervention_utils import *
from .model_utils import *
from .eval_utils import *


# Attention Activations
def gather_attn_activations(prompt_data, layers, dummy_labels, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing ICL prompt examples, and template information
    layers: layer names to get activatons from
    dummy_labels: labels and indices for a baseline prompt with the same number of example pairs
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    idx_map: map of token indices to respective averaged token indices
    idx_avg: dict containing token indices of multi-token words
    """   
    
    # Get sentence and token labe

In [7]:
# Continue reading extract_utils.py
print(extract_utils_content[5000:10000])

 stack_initial = torch.vstack([split_activations_by_head(activations_td[layer].input, model_config) for layer in model_config['attn_hook_names']]).permute(0,2,1,3)
        stack_filtered = stack_initial[:,:,list(idx_map.keys())]
        for (i,j) in idx_avg.values():
            stack_filtered[:,:,idx_map[i]] = stack_initial[:,:,i:j+1].mean(axis=2) # Average activations of multi-token words across all its tokens
        
        activation_storage[n] = stack_filtered

    mean_activations = activation_storage.mean(dim=0)
    return mean_activations

# Layer Activations
def gather_layer_activations(prompt_data, layers, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing
    layers: layer names to get activatons from
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    """   
    
    # Get sentence and token labels
    query = prompt

In [8]:
# Continue reading extract_utils.py - the compute_universal_function_vector function
print(extract_utils_content[10000:])

[i][1] for i in range(model_config['n_layers'])]) # (layers, heads, tokens, head_dim)
    value_weighted_attn = torch.einsum("abcd,abd->abcd", attentions, values.norm(dim=-1))
    return attentions, value_weighted_attn

def get_token_averaged_attention(dataset, model, model_config, tokenizer, n_shots=10, storage_max=100, filter_set=None):
    """

    Parameters:
    dataset: ICL dataset
    model: huggingface model
    model_config: dict with model information - n_layers, n_heads, etc.
    tokenizer: huggingface tokenizer
    n_shots: number of ICL example pairs to use for each prompt
    storage_max: max number of sentences to average attention pattern over
    filter_set: list of ints to filter to desired dataset instances

    Returns:
    attn_storage: attention heatmaps
    vw_attn_storage: value-weighted attention heatmaps
    token_labels: sample token labels for an n-shot prompt
    """
    if filter_set is not None:
        storage_size = min(len(filter_set), storage_max)
   

In [9]:
# Read the intervention utils to understand FV intervention
with open(os.path.join(repo_root, 'src/utils/intervention_utils.py'), 'r') as f:
    intervention_content = f.read()
print("=== intervention_utils.py ===")
print(intervention_content)

=== intervention_utils.py ===
from baukit import TraceDict, get_module
import torch
import re
import bitsandbytes as bnb

def get_module(model, name):
    """
    Finds the named module within the given model.
    """
    for n, m in model.named_modules():
        if n == name:
            return m
    raise LookupError(name)


def replace_activation_w_avg(layer_head_token_pairs, avg_activations, model, model_config, idx_map, batched_input=False, last_token_only=False):
    """
    An intervention function for replacing activations with a computed average value.
    This function replaces the output of one (or several) attention head(s) with a pre-computed average value 
    (usually taken from another set of runs with a particular property).
    The batched_input flag is used for systematic interventions where we are sweeping over all attention heads for a given (layer,token)
    The last_token_only flag is used for interventions where we only intervene on the last token (such as zero

In [10]:
# Read the model_utils.py
with open(os.path.join(repo_root, 'src/utils/model_utils.py'), 'r') as f:
    model_utils_content = f.read()
print("=== model_utils.py ===")
print(model_utils_content)

=== model_utils.py ===
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM
import os
import random
from typing import *


def load_gpt_model_and_tokenizer(model_name:str, device='cuda', revision=None):
    """
    Loads a huggingface model and its tokenizer

    Parameters:
    model_name: huggingface name of the model to load (e.g. GPTJ: "EleutherAI/gpt-j-6B", or "EleutherAI/gpt-j-6b")
    device: 'cuda' or 'cpu'
    
    Returns:
    model: huggingface model
    tokenizer: huggingface tokenizer
    MODEL_CONFIG: config variables w/ standardized names
    
    """
    assert model_name is not None

    print("Loading: ", model_name)

    if model_name == 'gpt2-xl':
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

        MODEL_CONFIG={"n_heads":model.config.n_head,
        

In [11]:
# Read the prompt_utils.py
with open(os.path.join(repo_root, 'src/utils/prompt_utils.py'), 'r') as f:
    prompt_utils_content = f.read()
print("=== prompt_utils.py ===")
print(prompt_utils_content[:8000])

=== prompt_utils.py ===
import numpy as np
import pandas as pd
from pathlib import Path
import os
from typing import *
from sklearn.model_selection import train_test_split



def create_fewshot_primer(prompt_data) -> str:
    """Creates the primer string for GPT in-context learning
    
    Parameters:
    prompt_data: dict containing ICL prompt examples, and template information

    Returns:
    prompt: the constructed ICL prompt primer as a string
    """       
    prompt = ''
    prompt += prompt_data['prefixes']['instructions'] + prompt_data['instructions'] + prompt_data['separators']['instructions']
    
    for example in prompt_data['examples']:
        
        prompt += prompt_data['prefixes']['input'] + example['input'] + prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output'] + example['output'] + prompt_data['separators']['output']
        
    return prompt
    
def create_prompt(prompt_data, sentence=None) -> str:
    """Creates a prompt u

In [12]:
# Continue reading prompt_utils.py
print(prompt_utils_content[8000:])

l_labels

def tokenize_labels(sentence_parts, text_labels, tokenizer, prepend_bos=False):
    """
    Extends phrase-level labels across tokenization for in-context learning prompts. Tested with GPT-2's tokenizer from huggingface.
    Parameters:
    sentence_parts: list, where each element is either a token (str), phrase (str), or list of tokens/phrases
    text_labels: list with the same structure as 'sentence_parts', with a corresponding label for that level of the input sentence.
    tokenizer: huggingface tokenizer
    
    Returns: 
    labels: flattened/extended list of token labels for an ICL prompt (split into parts, contained in sentence_parts and text_labels)

    based on the tokenize_and_preserve_labels function from:
    https://www.depends-on-the-definition.com/named-entity-recognition-with-bert/
    """
    
    # If the model typically prepends a bos, we add a bos label to label init
    if prepend_bos:
        labels = extend_labels(sentence_parts, text_labels, tokeni

In [13]:
# Read eval_utils.py
with open(os.path.join(repo_root, 'src/utils/eval_utils.py'), 'r') as f:
    eval_utils_content = f.read()
print("=== eval_utils.py ===")
print(eval_utils_content)

=== eval_utils.py ===
import torch
import string
import itertools
import re
import numpy as np
from tqdm import tqdm
from .prompt_utils import *
from .model_utils import *
from .intervention_utils import *


def compute_top_k_accuracy(target_token_ranks, k=10) -> float:
    """
    Evaluation to compute topk accuracy.

    Parameters:
    target_token_ranks: the distribution of output token ranks
    k: how many tokens we're looking at (top K)

    Return:
    The accuracy of the token in the top k of tokens
    """

    target_token_ranks = np.array(target_token_ranks)
    return (target_token_ranks < k).sum(axis=0) / len(target_token_ranks) 

def compute_individual_token_rank(prob_dist, target_id) -> int:
    """
    Individual computation of token ranks across a single distribution.

    Parameters:
    prob_dist: the distribution of scores for a single output
    target_id: the target id we care about

    Return:
    A single value representing the token rank for that single token

In [14]:
# Create the evaluation/replications directory if it doesn't exist
import os
from datetime import datetime

repo_root = '/net/scratch2/smallyan/function_vectors_eval'
replication_dir = os.path.join(repo_root, 'evaluation', 'replications')
os.makedirs(replication_dir, exist_ok=True)
print(f"Created replication directory: {replication_dir}")
print(f"Contents: {os.listdir(replication_dir) if os.path.exists(replication_dir) else 'empty'}")

Created replication directory: /net/scratch2/smallyan/function_vectors_eval/evaluation/replications
Contents: []


# Function Vectors Replication Notebook

This notebook replicates the key experiments from the "Function Vectors in Large Language Models" paper.

## Overview

The paper investigates whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning.

## Key Components to Replicate:
1. Load model and tokenizer
2. Load dataset and compute task-conditioned mean activations
3. Compute function vector (FV) from top causal attention heads
4. Test FV intervention in different contexts (ICL, shuffled-label, zero-shot, natural text)

In [15]:
# Check CUDA availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA H200 NVL
GPU Memory: 150.11 GB


## Setup and Imports

We'll reimplement the core functionality without copying code verbatim from the repository.

In [16]:
# Install required packages
import subprocess
subprocess.run(['pip', 'install', 'baukit', '-q'], check=True)
print("baukit installed")

baukit installed


In [17]:
# Core imports
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Any
from sklearn.model_selection import train_test_split
from pathlib import Path

# Disable gradient computation for inference
torch.set_grad_enabled(False)

print("All imports successful!")

All imports successful!


In [18]:
# Utility function for setting seeds for reproducibility
def set_random_seed(seed: int) -> None:
    """Set seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

set_random_seed(42)
print("Seed set to 42")

Seed set to 42


## Dataset Utilities

Reimplementing the dataset loading and prompt creation utilities.

In [19]:
# Dataset class for ICL experiments
class ICLDataset:
    """Dataset class for in-context learning experiments with input-output pairs."""
    
    def __init__(self, data):
        if isinstance(data, str):
            self.data = pd.read_json(data)
        elif isinstance(data, dict):
            self.data = pd.DataFrame(data)
        else:
            self.data = data
        self.data = self.data[['input', 'output']]
    
    def __getitem__(self, idx):
        if isinstance(idx, int):
            return self.data.iloc[idx].to_dict()
        elif isinstance(idx, slice) or isinstance(idx, (list, np.ndarray)):
            return self.data.iloc[idx].to_dict(orient='list')
        elif isinstance(idx, str):
            return self.data[idx].tolist()
        raise ValueError(f"Invalid index type: {type(idx)}")
    
    def __len__(self):
        return len(self.data)
    
    def __repr__(self):
        return f"ICLDataset(features={self.data.columns.tolist()}, num_rows={len(self)})"


def split_dataset(dataset: ICLDataset, test_size: float = 0.3, seed: int = 42) -> Dict[str, ICLDataset]:
    """Split dataset into train, valid, and test sets."""
    train_data, valid_data = train_test_split(dataset.data, test_size=test_size, random_state=seed)
    test_data, valid_data = train_test_split(valid_data, test_size=test_size, random_state=seed)
    
    return {
        'train': ICLDataset(train_data.to_dict(orient='list')),
        'valid': ICLDataset(valid_data.to_dict(orient='list')),
        'test': ICLDataset(test_data.to_dict(orient='list'))
    }


def load_task_dataset(task_name: str, data_root: str = '/net/scratch2/smallyan/function_vectors_eval/dataset_files',
                      test_size: float = 0.3, seed: int = 32) -> Dict[str, ICLDataset]:
    """Load a task dataset from the dataset files."""
    for folder in ['abstractive', 'extractive']:
        file_path = os.path.join(data_root, folder, f'{task_name}.json')
        if os.path.exists(file_path):
            dataset = ICLDataset(file_path)
            return split_dataset(dataset, test_size=test_size, seed=seed)
    raise FileNotFoundError(f"Dataset {task_name} not found in {data_root}")


# Test dataset loading
dataset = load_task_dataset('antonym')
print(f"Train: {len(dataset['train'])}, Valid: {len(dataset['valid'])}, Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]}")

Train: 1678, Valid: 216, Test: 504
Sample: {'input': 'hardware', 'output': 'software'}


In [20]:
# Prompt construction utilities
def build_prompt_data(word_pairs: Dict[str, List[str]], 
                      query_pair: Optional[Dict[str, str]] = None,
                      add_bos: bool = True,
                      shuffle_outputs: bool = False,
                      prefixes: Dict[str, str] = None,
                      separators: Dict[str, str] = None) -> Dict:
    """
    Build prompt data structure for ICL experiments.
    
    Args:
        word_pairs: Dict with 'input' and 'output' lists
        query_pair: Single input-output pair for the query
        add_bos: Whether to prepend BOS token
        shuffle_outputs: Whether to shuffle the output labels
        prefixes: Custom prefixes for input/output/instructions
        separators: Custom separators for input/output/instructions
    """
    if prefixes is None:
        prefixes = {"input": "Q:", "output": "A:", "instructions": ""}
    if separators is None:
        separators = {"input": "\n", "output": "\n\n", "instructions": ""}
    
    # Add BOS token to instruction prefix if needed
    if add_bos:
        prefixes = {k: (v if k != 'instructions' else '<|endoftext|>' + v) 
                   for k, v in prefixes.items()}
    
    # Handle query pair
    if query_pair is not None:
        query_pair = {k: (v[0] if isinstance(v, list) else v) for k, v in query_pair.items()}
    
    # Build examples
    inputs = word_pairs.get('input', [])
    outputs = word_pairs.get('output', [])
    
    if shuffle_outputs and len(outputs) > 0:
        outputs = np.random.permutation(outputs).tolist()
    
    # Add space prefix to tokens for proper tokenization
    examples = [{'input': ' ' + str(inp), 'output': ' ' + str(out)} 
                for inp, out in zip(inputs, outputs)]
    
    query_with_space = None
    if query_pair:
        query_with_space = {k: ' ' + str(v) for k, v in query_pair.items()}
    
    return {
        'instructions': '',
        'prefixes': prefixes,
        'separators': separators,
        'examples': examples,
        'query_target': query_with_space
    }


def construct_prompt(prompt_data: Dict, query: str = None) -> str:
    """Construct the full ICL prompt string from prompt data."""
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    
    if isinstance(query, list):
        query = query[0]
    
    # Build primer (few-shot examples)
    prompt = prompt_data['prefixes']['instructions'] + prompt_data['instructions'] + prompt_data['separators']['instructions']
    
    for ex in prompt_data['examples']:
        prompt += prompt_data['prefixes']['input'] + ex['input'] + prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output'] + ex['output'] + prompt_data['separators']['output']
    
    # Add query
    prompt += prompt_data['prefixes']['input'] + query + prompt_data['separators']['input']
    prompt += prompt_data['prefixes']['output']
    
    return prompt


# Test prompt construction
test_pairs = dataset['train'][:5]
test_query = dataset['test'][21]
prompt_data = build_prompt_data(test_pairs, query_pair=test_query, add_bos=True)
prompt_str = construct_prompt(prompt_data)
print("ICL Prompt:")
print(repr(prompt_str[:500]))

ICL Prompt:
'<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:'


## Model Loading

Load GPT-J 6B model, which is the smallest model used in the original experiments.

In [21]:
# Model configuration and loading
def load_model_and_tokenizer(model_name: str, device: str = 'cuda'):
    """
    Load a language model and its tokenizer with proper configuration.
    
    Returns:
        model: HuggingFace model
        tokenizer: HuggingFace tokenizer
        config: Dictionary with model-specific configuration
    """
    print(f"Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True).to(device)
    
    # Build model configuration
    if 'gpt-j' in model_name.lower():
        config = {
            'n_heads': model.config.n_head,
            'n_layers': model.config.n_layer,
            'resid_dim': model.config.n_embd,
            'name_or_path': model.config.name_or_path,
            'attn_hook_names': [f'transformer.h.{layer}.attn.out_proj' for layer in range(model.config.n_layer)],
            'layer_hook_names': [f'transformer.h.{layer}' for layer in range(model.config.n_layer)],
            'prepend_bos': False  # GPT-J doesn't auto-prepend BOS
        }
    else:
        raise NotImplementedError(f"Model {model_name} not yet supported")
    
    return model, tokenizer, config


# Load GPT-J
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_model_and_tokenizer(model_name)
print(f"\nModel config:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden dim: {model_config['resid_dim']}")

# Set the edit layer (typically around L/3 for GPT-J, which is layer 9)
EDIT_LAYER = 9
print(f"\nEdit layer for FV intervention: {EDIT_LAYER}")

Loading model: EleutherAI/gpt-j-6b


In [22]:
# Check if model loaded successfully
print(f"Model loaded on device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [23]:
# Continue verifying model loaded
print("Model verification...")
try:
    print(f"Model loaded on device: {model.device}")
    print(f"Model dtype: {model.dtype}")
except Exception as e:
    print(f"Error: {e}")

In [24]:
# Wait for model to load - simple check
import time
time.sleep(2)
print("Checking model status...")
print(f"Model: {type(model)}")
print(f"Device: {model.device}")

In [25]:
# Simple test
print("Test")

In [26]:
print("Testing output visibility")
x = 1 + 1
print(f"Result: {x}")